# Healthcare Patient Deterioration Analytics
**Student:** Vansh Jain \
**Technologies Used:** Azur Data Factory · Databricks PySpark · Unity Catalog · Spark MLlib \
**Notebook:** Export Layer\
**Layers:** Bronze → Silver → Gold

### Storage Account Configuration

In [0]:
storage_account = 'healthcaremedalliondev'
storage_key = '<>'

In [0]:
spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

In [0]:
gold_path = f'abfss://gold@{storage_account}.dfs.core.windows.net/'

## Exporting KPI delta tables to csv's

In [0]:
gold_patient_summary = spark.read.format('delta').load(
    gold_path + 'gold_patient_summary'
)

In [0]:
(
    gold_patient_summary.write
    .mode('overwrite')
    .option('header', True)
    .csv(f'abfss://gold@{storage_account}.dfs.core.windows.net/csv/gold_patient_summary')
)

In [0]:
gold_hourly_vitals = spark.read.format('delta').load(
    gold_path + 'gold_hourly_vitals'
)

In [0]:
(
    gold_hourly_vitals.write
    .mode('overwrite')
    .option('header', True)
    .csv(f'abfss://gold@{storage_account}.dfs.core.windows.net/csv/gold_hourly_vitals')
)

In [0]:
gold_admission_analysis = spark.read.format('delta').load(
    gold_path + 'gold_admission_analysis'
)

In [0]:
(
    gold_admission_analysis.write
    .mode('overwrite')
    .option('header', True)
    .csv(f'abfss://gold@{storage_account}.dfs.core.windows.net/csv/gold_admission_analysis')
)

In [0]:
gold_age_group_analysis = spark.read.format('delta').load(
    gold_path + 'gold_age_group_analysis'
)

In [0]:
(
    gold_age_group_analysis.write
    .mode('overwrite')
    .option('header', True)
    .csv(f'abfss://gold@{storage_account}.dfs.core.windows.net/csv/gold_age_group_analysis')
)

In [0]:
gold_sepsis_analysis = spark.read.format('delta').load(
    gold_path + 'gold_sepsis_analysis'
)

In [0]:
(
    gold_sepsis_analysis.write
    .mode('overwrite')
    .option('header', True)
    .csv(f'abfss://gold@{storage_account}.dfs.core.windows.net/csv/gold_sepsis_analysis')
)

In [0]:
gold_oxygen_analysis = spark.read.format('delta').load(
    gold_path + 'gold_oxygen_analysis'
)

In [0]:
(
    gold_oxygen_analysis.write
    .mode('overwrite')
    .option('header', True)
    .csv(f'abfss://gold@{storage_account}.dfs.core.windows.net/csv/gold_oxygen_analysis')
)

In [0]:
gold_high_risk_patients = spark.read.format('delta').load(
    gold_path + 'gold_high_risk_patients'
)

In [0]:
(
    gold_high_risk_patients
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(f'abfss://gold@{storage_account}.dfs.core.windows.net/csv/gold_high_risk_patients')
)

In [0]:
gold_ml_predictions = spark.read.format('delta').load(
    gold_path + 'gold_ml_predictions'
)

In [0]:
from pyspark.sql.functions import udf, col
from pyspark.sql.types import DoubleType

In [0]:
get_probability = udf(
    lambda x: float(x[1]),
    DoubleType()
)

In [0]:
prediction_export = (
    gold_ml_predictions
    .withColumn(
        "deterioration_probability",
        get_probability(col("probability"))
    )
    .drop("probability")
)

In [0]:
(
    prediction_export
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(f"abfss://gold@{storage_account}.dfs.core.windows.net/csv/gold_ml_predictions")
)